# 13장. Hooks와 Telemetry로 실행 관측하기

OpenAI-hosted CSV 보고서 실행에 애플리케이션 관측을 연결합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | 실제 API callback·OTel span·선택 Trace 조사 |
| 이번 실습 조합 | OpenAI-hosted 기본 실행 + 메모리 OTel · Collector/로컬 Hook은 선택 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · 읽기용 · 실습은 로컬 Python·선택 Docker


## 학습 안내

CSV 보고서 실행에서 무엇을 관측할지 선택하는 실습입니다. 실제 API 이벤트를 애플리케이션 callback과 OTel span으로 연결하고, API 기본 Trace는 필요할 때 Dashboard 또는 선택 export로 확인합니다. 이전 장의 세션·커널에 의존하지 않습니다.

| 알고 싶은 내용 | 확인할 기록 |
| --- | --- |
| 모델이 어떤 도구/명령을 실행했는가? | API 기본 Trace·Items |
| 애플리케이션이 어떤 이벤트를 처리했는가? | callback records |
| 실제 span이 생성·전달됐는가? | 메모리 OTel exporter 또는 Collector |
| SessionStart 입력 형식은 어떻게 처리하는가? | 별도 로컬 Hook 프로토콜 재생 |

기본은 API+메모리 OTel이며 Docker와 Trace export는 선택입니다. native Codex dispatch·실제 청구·Dashboard 화면 확인은 별도 검증 범위입니다. 기본 Trace와 app span을 같은 기록으로 해석하지 않습니다. [실습 안내와 검증 범위](../../docs/validation.md)


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실행 모드

로컬 uv 환경에서 위에서 아래로 실행합니다. 세 실행 스위치와 EXPORT_TRACES는 기본 False입니다. Colab은 읽기용이며 실습은 로컬 Python·선택 Docker에서 수행합니다.

| 경로 | RUN_LIVE | RUN_DOCKER | RUN_LOCAL_CHECKS |
| --- | --- | --- | --- |
| API + 메모리 OTel | True | False | False |
| API + Collector만 | True | True | False |
| 별도 로컬 실습만 | False | True | True |

EXPORT_TRACES는 위 API 경로에서 독립적으로 선택합니다. 로컬 실습만 할 때는 API 준비와 6단계를 건너뜁니다. 두 경로를 켜면 API 정리 후 별도 worker에서 실습합니다.

```bash
uv sync --locked --group telemetry
# Live API with in-memory application spans.
uv run --group dev python scripts/run_deployment.py --chapter 13 --live
# Live API with a Collector, without local Hook replay.
uv run --group dev python scripts/run_deployment.py --chapter 13 --live --docker
# Separate local checks; no Agents API calls.
uv run --group dev python scripts/run_deployment.py --chapter 13 --docker --build
# Optional API Trace export; not another model task.
TUTORIAL_EXPORT_TRACES=1 uv run --group dev python scripts/run_deployment.py --chapter 13 --live
```

CLI의 --docker만 켜면 실행 사본의 RUN_LOCAL_CHECKS를 켭니다. --live --docker는 Collector만 사용합니다. 오프라인 검증은 API·Docker를 모두 제외합니다.


직접 Jupyter에서 메모리 관측을 실습하려면 `uv sync --locked --group telemetry`로 설치한 뒤 `uv run --group telemetry jupyter lab`으로 실행합니다. 선택 Collector 전송은 `uv run --group telemetry-otlp jupyter lab`을 사용합니다. 자동 검증 명령의 dev 그룹은 관측·Collector 검사 의존성도 포함합니다. Colab 실행본에는 메모리 관측 SDK만 추가하며 HTTP exporter는 설치하지 않습니다. Colab에서 12–14장은 계속 읽기용입니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


### 관측을 위한 라이브러리 역할

API 호출은 원본 셀에 남기며 helper는 Docker 준비·OTLP 전송·JSONL 해석·자원 종료를 담당합니다. ApplicationHooks는 호출자 코드의 이벤트 adapter이며 API native Hook 설정이나 모델 실행 recorder가 아닙니다.


In [ ]:
import json
from contextlib import ExitStack
from tutorial_helpers.common import load_settings
from tutorial_helpers.deployment import (
    DockerLab,
    build_images,
    docker,
    SKILL_COMMAND,
    ApplicationHooks,
    start_collector,
    read_collector_file,
    decode_otlp_spans,
)


### 입력 바이트와 선택 설정

CSV는 원본 바이트로 읽어 그대로 전송합니다. 선택 export의 시도 횟수·간격은 준비 단계에서 검사하며, 개별 HTTP timeout과 전체 소요 시간은 다릅니다. usage가 없으면 미보고로 유지하고 0을 만들어 채우지 않습니다.


In [ ]:
RUN_DOCKER = False
RUN_LIVE = False
RUN_LOCAL_CHECKS = False
MODEL_OVERRIDE = ""
CHAPTER = "13_hooks_telemetry"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
root = settings.root
offline_check = os.environ.get("TUTORIAL_OFFLINE_CHECK") == "1"
docker_enabled = RUN_DOCKER and (not offline_check)
csv_bytes = (root / "data/sales.csv").read_bytes()
csv_text = csv_bytes.decode("utf-8")
EXPORT_TRACES = False
TRACE_EXPORT_ATTEMPTS = 6
TRACE_EXPORT_INTERVAL_SECONDS = 2

if EXPORT_TRACES and (TRACE_EXPORT_ATTEMPTS < 1 or TRACE_EXPORT_INTERVAL_SECONDS < 0):
    raise ValueError("Invalid trace export retry settings")
print({"api_enabled": settings.enabled, "docker_enabled": docker_enabled,
       "local_checks": RUN_LOCAL_CHECKS, "export_traces": EXPORT_TRACES})


## 1. API 요청 준비

API 요청·스트리밍·파일 다운로드·자원 정리는 직접 실행합니다. 정답 비교와 검증 기록은 별도 검증 도구에서 처리합니다.


In [ ]:
from tutorial_helpers.common import new_session_metadata, cleanup_resources, cleanup_on_error, safe_error, check_event
from uuid import uuid4
import base64
import time


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)


## 2. API 실행과 관측을 단계별로 구분하기

```text
Agents API events → event.type branch → hooks.emit → application OTel span
                                            ↘ callback records
Agents API default Trace → Dashboard / optional Trace export
```

hooks.records는 애플리케이션의 처리 기록이며 전송 증거는 아닙니다. 실제 메모리 exporter/Collector span도 따로 읽습니다. app OTel Trace ID와 API Trace ID가 자동으로 같아지지 않으며 Session·Turn 속성으로 연결해 읽습니다.

각 hooks.emit은 짧은 이벤트 span입니다. span 지속 시간이 모델 작업 전체 시간은 아닙니다. duration_ms 속성은 API 생성·스트리밍 시간이며 셀 사이 검토·다운로드 시간을 제외합니다. status 속성과 OTel span의 내부 status code도 다른 값입니다.

app 허용 필드 필터는 callback/span에만 적용합니다. API 기본 Trace나 export를 정제하지 않습니다. API Trace에는 기록된 입력·출력이 포함될 수 있습니다. [공식 Tracing](https://developers.openai.com/api/docs/guides/agents-api/tracing)

순서: 관측 준비 → API 실행 → 보고서 읽기 → 실제 app span 확인 → 선택 Trace 확인/export → 정리. 중간에 멈춰도 6번 정리 셀을 실행하세요.


In [ ]:
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [
        {
            "type": "inline",
            "path": "/workspace/sales.csv",
            "data": base64.b64encode(csv_bytes).decode("ascii"),
        }
    ],
}
task = (
    "Use Python's standard csv module to read /workspace/sales.csv. "
    "Write /workspace/outputs/summary.json in the requested format and "
    "/workspace/outputs/report.md in Korean starting with '# 매출 분석 보고서'. "
    "Include row count, total quantity, total revenue as digits, and each product's revenue. "
    "Read both files back. " + METRICS_CONTRACT
)


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

새 client로 한 번만 초기화합니다. 진행 중 다시 초기화하면 세션/Collector 식별자를 잃습니다. 1단계는 상태를 초기화하지 않고 관측 자원 중복 준비를 차단하며 2단계는 같은 실행의 세션 생성을 한 번만 허용합니다. 오류/정리 후에는 새 client·상태로 시작합니다.


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/deployment-api" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
resources = ExitStack()
hooks = transport = collector = result = None
trace_export, trace_count, trace_export_error = "not_requested", 0, None
report_downloaded = application_hooks_checked = False

downloads = turn = None
observed_spans = []
telemetry_prepared = False
trace_span_count = 0
trace_export_error_stage = None


### 1. 관측 자원 준비

앞에서 초기화한 실행 상태를 사용해 메모리 OTel provider를 준비합니다. Docker 모드라면 Collector도 시작하며 아직 Agent는 실행하지 않습니다. 같은 실행에서 자원 준비는 한 번만 허용합니다. 중간에 멈추면 6번 정리 셀을 실행하세요.


In [ ]:
with cleanup_on_error(
    client, sessions, agents, attempts, errors=cleanup_errors,
    finalizers=(resources.close,),
):
    if telemetry_prepared:
        raise RuntimeError("Telemetry is already prepared; do not allocate resources twice")
    endpoint = None
    if docker_enabled:
        transport = resources.enter_context(DockerLab("13_api_telemetry", root))
        network = transport.network()
        collector, endpoint = start_collector(transport, network)
        endpoint += "/v1/traces"
    hooks = ApplicationHooks(endpoint, run_id)
    resources.callback(hooks.close)
    telemetry_prepared = True
    print("[1/6] Telemetry ready:", "Docker OTLP Collector" if docker_enabled else "in-memory")
    print("Local records:", output_dir / "application-hooks.json")


### 실패 기록을 읽는 기준

API가 root Turn 실패 이벤트를 보낸 경우에만 turn.failed로 기록합니다. 로컬 API 호출/파일 처리 오류는 report.failed의 local_error이며 원격 작업 실패로 단정하지 않습니다. 관찰 timeout도 원격 실패 증명이 아닙니다. 실패 기록 생성에 다시 오류가 나더라도 원래 예외와 정리를 보존합니다. 오류 body나 프롬프트는 span에 추가하지 않습니다.


### 2. Agent 실행·이벤트 연결

별도 사용자 입력 없이 API의 분석 결과를 기다립니다. API가 보낸 이벤트를 event.type으로 분기하고 session.created·root turn.completed를 hooks.emit으로 별도 기록하는 지점을 확인하세요. 텍스트 delta는 표시하지만 Telemetry 본문으로 보내지 않습니다.


In [ ]:
with cleanup_on_error(
    client, sessions, agents, attempts, errors=cleanup_errors,
    finalizers=(resources.close,),
):
    try:
        if not telemetry_prepared or hooks.closed or attempts:
            raise RuntimeError("Prepare telemetry first and create this session only once")
        print("[2/6] Running the agent and recording selected application events...", flush=True)
        started = time.monotonic()
        hooks.emit("report.started", chapter=13, status="running")
        # 1. Create a session and stream the response
        with client.beta.agents.sessions.create(
            agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
            environment=environment,
            input=task,
            metadata=new_session_metadata(
                CHAPTER, run_id, attempts=attempts, output_dir=output_dir
            ),
            stream=True,
        ).with_result_collection() as stream:
            for event in stream:
                if event.type == "agent.session.created":
                    sessions[event.session.id] = {
                        "deleted": False,
                        "environment_id": getattr(event.session.environment, "id", None),
                    }
                    print("Session:", event.session.id)
                    hooks.emit("session.created", chapter=13, session_id=event.session.id)
                elif event.type == "agent.session.turn.output_text.delta":
                    print(event.delta, end="", flush=True)
                elif (
                    event.type == "agent.session.turn.completed" and event.turn.subagent_id is None
                ):
                    hooks.emit(
                        "turn.completed",
                        chapter=13,
                        session_id=event.session_id,
                        turn_id=event.turn.id,
                        status=event.turn.status,
                    )
                elif event.type == "agent.session.turn.failed" and event.turn.subagent_id is None:
                    hooks.emit("turn.failed", chapter=13, session_id=event.session_id,
                               turn_id=event.turn.id, status=event.turn.status)
                    check_event(event)
                elif event.type == "agent.session.requires_action":
                    raise RuntimeError("Unexpected external action in this hosted exercise")
                else:
                    check_event(event)
                if time.monotonic() - started > 600:
                    raise TimeoutError("Event-observed execution deadline exceeded")
            result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})
        api_duration_ms = int((time.monotonic() - started) * 1000)
        print("\nAgent turn returned:", result.turn_id)
    except BaseException:
        try:
            hooks.emit("report.failed", chapter=13, status="local_error")
        except Exception:
            pass
        raise


### 3. 실행 결과 확인·다운로드

완료된 Turn과 usage를 조회하고 보고서를 다운로드합니다. 출력된 파일을 열어 검토한 뒤 다음 셀로 진행하세요. usage=None은 미보고이며 0이 아닙니다. duration_ms는 2번 셀의 세션 생성·스트리밍에 걸린 시간으로, 셀 사이 검토와 다운로드 시간은 제외합니다.


In [ ]:
with cleanup_on_error(
    client, sessions, agents, attempts, errors=cleanup_errors,
    finalizers=(resources.close,),
):
    try:
        report_downloaded = False
        if result is None:
            raise RuntimeError("Complete the API run before downloading")
        print("[3/6] Checking the turn and downloading the report...", flush=True)
        turn = client.beta.agents.sessions.turns.retrieve(
            result.turn_id, session_id=result.session_id, timeout=15
        )
        if (result.turn.status != "completed" or turn.status != "completed"
            or turn.id != result.turn_id or turn.subagent_id is not None):
            raise RuntimeError("Root turn completion was not confirmed")
        actual = json.loads(result.output_text)
        downloads = output_dir / "downloads"
        downloads.mkdir(parents=True, exist_ok=True)
        scoped = client.beta.agents.sessions.artifacts.for_result(result)
        for name in ("summary.json", "report.md"):
            artifact = scoped.download("/workspace/outputs/" + name, to=downloads / name, timeout=30)
            print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
                   "sandbox_path": artifact.path, "local_path": str(downloads / name)})
        report_downloaded = True
        usage = turn.usage.to_dict() if turn.usage else {}
        hooks.emit(
            "report.completed", chapter=13, session_id=result.session_id,
            turn_id=turn.id, status=turn.status, duration_ms=api_duration_ms,
            **{k: v for k, v in usage.items() if k != "reported"},
        )
        print("Turn status:", turn.status)
        print("Summary:", actual)
        print("Report:", downloads / "report.md")
        print("Turn usage:", turn.usage.to_dict() if turn.usage else None)
        print("API creation and streaming duration (ms):", api_duration_ms)
        downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
        print("Downloaded summary.json:", json.dumps(downloaded_summary, indent=2))
        from IPython.display import Markdown, display
        display(Markdown((downloads / "report.md").read_text(encoding="utf-8")))
    except BaseException:
        try:
            hooks.emit("report.failed", chapter=13, status="local_error")
        except Exception:
            pass
        raise


### 4. callback 기록과 실제 app span 읽기

callback의 이름·run/session/turn·상태·시간·usage를 읽고 실제 span과 비교합니다. flush는 전달을 비우며 provider를 닫지 않습니다. 종료는 6단계에서 수행합니다. 기본 경로에서는 실제 메모리 exporter를 읽고 Docker 경로에서는 Collector JSONL을 해석합니다.

app Trace ID·Span ID는 실제 OTel 식별자이며 API Trace ID와 자동으로 연결되는 값이 아닙니다. 이 실습은 session_id/turn_id 속성으로 상관관계를 확인합니다. 수신 span이 없으면 관측 성공으로 표시하지 않습니다. 이 단계는 API Trace export를 호출하지 않습니다.


In [ ]:
with cleanup_on_error(
    client, sessions, agents, attempts, errors=cleanup_errors,
    finalizers=(resources.close,),
):
    application_hooks_checked = False
    if not report_downloaded:
        raise RuntimeError("Complete the report before inspecting application spans")
    print("[4/6] Reading callback records and actual application spans...", flush=True)
    hooks.flush()
    print("Callback records:")
    for record in hooks.records:
        print(record)
    if docker_enabled:
        traces = read_collector_file(transport, collector, "traces.jsonl", needle="report.completed")
        spans = decode_otlp_spans(traces)
    else:
        spans = [dict(name=span.name, trace_id=f"{span.context.trace_id:032x}",
                      span_id=f"{span.context.span_id:016x}",
                      service_name=span.resource.attributes.get("service.name"),
                      attributes=dict(span.attributes))
                 for span in hooks.exporter.get_finished_spans()]
    observed_spans = [span for span in spans
                      if span["service_name"] == "tutorial.application-hooks"
                      and span["attributes"].get("run_id") == run_id]
    if not observed_spans:
        raise RuntimeError("No application spans were received for this run")
    print("Actual application spans:")
    for span in observed_spans:
        print(span)
    (output_dir / "application-hooks.json").write_text(json.dumps(hooks.records))
    (output_dir / "application-spans.json").write_text(json.dumps(observed_spans))
    application_hooks_checked = True
    print("Saved callback records and actual spans under:", output_dir)
    print("The provider stays open until cleanup. These spans differ from API Trace.")


### 5. 선택: API 기본 Trace 확인·export

EXPORT_TRACES=False가 기본이며 Dashboard 주소·Session ID만 표시하고 Trace API는 호출하지 않습니다. 확인할 때는 이 셀 뒤 정리 전에 Logs → Agents에서 같은 프로젝트·Session·Turn을 찾으세요. 사용자 input 대기는 넣지 않습니다.

True이면 Trace API를 조회하며 limit=20은 페이지 크기이고 SDK 반복은 다음 페이지도 읽습니다. timeout=15는 개별 요청 제한이고 재시도 횟수/간격이 전체 실행 시간을 보장하지는 않습니다. 조회는 새 모델 작업을 제출하지 않습니다. export에는 조직 활성화와 프로젝트 읽기 권한이 필요하고 Trace는 답변보다 늦게 준비될 수 있습니다. [공식 export](https://developers.openai.com/api/docs/guides/agents-api/tracing#export-session-traces)

| 상태 | 해석 |
| --- | --- |
| not_requested | export 미선택, 기본 실습에서 정상 |
| exported | 실제 span을 포함한 OTLP 저장 완료 |
| not_ready | 제한된 조회에서 데이터/span 미확보; 부재·0 usage로 단정하지 않음 |
| unavailable | 조회 또는 저장 오류; 오류 단계/종류 확인 |

Trace 객체 수와 실제 span 수를 따로 표시합니다. 파일은 임시 저장 후 교체하며 이전 성공 파일은 후속 오류로 삭제하지 않습니다. 현재 상태와 마지막 성공 파일을 구분하세요. export/화면 확인 결과는 보고서 분석 성공과 별개입니다. 모든 경우 정리 셀은 실행합니다. app 필터는 API Trace/export의 입력·출력을 정제하지 않습니다.


In [ ]:
with cleanup_on_error(
    client, sessions, agents, attempts, errors=cleanup_errors,
    finalizers=(resources.close,),
):
    print("[5/6] Optional Agents API Trace inspection/export...", flush=True)
    print("Dashboard: https://platform.openai.com/logs?api=agents")
    print("Session ID:", result.session_id)
    trace_export, trace_count, trace_span_count = "not_requested", 0, 0
    trace_export_error = trace_export_error_stage = None
    if EXPORT_TRACES:
        temporary = output_dir / "traces.otlp.json.tmp"
        error_stage = "query"
        try:
            resources_to_export = []
            for attempt in range(TRACE_EXPORT_ATTEMPTS):
                print(f"Trace export attempt {attempt + 1}/{TRACE_EXPORT_ATTEMPTS}", flush=True)
                published = [trace.to_dict() for trace in client.beta.agents.sessions.traces.list(
                    result.session_id, order="asc", limit=20, timeout=15,
                )]
                trace_count = len(published)
                resources_to_export = [resource for trace in published
                                       for resource in trace["otlp"].get("resourceSpans", [])]
                trace_span_count = sum(len(scope.get("spans", [])) for resource in resources_to_export
                                       for scope in resource.get("scopeSpans", []))
                if trace_span_count:
                    break
                if attempt + 1 < TRACE_EXPORT_ATTEMPTS:
                    time.sleep(TRACE_EXPORT_INTERVAL_SECONDS)
            if trace_span_count:
                error_stage = "storage"
                temporary.write_text(json.dumps({"resourceSpans": resources_to_export}))
                temporary.replace(output_dir / "traces.otlp.json")
                trace_export = "exported"
                print("Export file:", output_dir / "traces.otlp.json")
            else:
                trace_export = "not_ready"
        except Exception as trace_error:
            trace_export, trace_export_error = "unavailable", type(trace_error).__name__
            trace_export_error_stage = error_stage
        finally:
            try:
                temporary.unlink(missing_ok=True)
            except OSError:
                pass
    print({"trace_export": trace_export, "trace_objects": trace_count,
           "actual_spans": trace_span_count, "error_stage": trace_export_error_stage,
           "error_type": trace_export_error})
    print("Export status does not determine report analysis success.")
    print("Inspect the Dashboard now if needed, then run stage 6 to clean up.")


### 6. 자원 정리·최종 요약

마지막에 반드시 실행하세요. 일부 단계만 마친 경우에도 세션·OTel·선택 Docker 자원을 정리합니다. 정상 경로에서는 4번 셀에서 Collector를 확인한 뒤 종료하고, 검증 실행기는 이 셀에서 삭제 전에 결과 검사·Docker 기록을 수행합니다. 오류·사용자 중단 시 앞선 단계가 정리를 시도하지만 강제 커널 종료까지 보장하지는 않습니다. 오류 후에는 상태를 확인하고 새 실행을 준비하세요.


In [ ]:
try:
    print("[6/6] Cleaning up the session, OTel provider and optional Docker resources...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    finally:
        try:
            resources.close()
        except Exception as resource_error:
            cleanup_errors.append(safe_error(resource_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(value.get("verified_absent") for value in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({
    "api_status": result.turn.status if result is not None else "not_completed",
    "report_downloaded": report_downloaded,
    "application_hooks_checked": application_hooks_checked,
    "trace_export": trace_export,
    "session_cleanup": "verified" if sessions else "not_needed",
    "otel_closed": hooks.closed if hooks is not None else "not_created",
})


## 3. 선택 로컬 실습: OTLP와 Hook 프로토콜 재생

RUN_DOCKER와 RUN_LOCAL_CHECKS가 모두 True일 때만 실행합니다. API+Collector 경로와 별도로 worker의 Skill callback을 전송하고 SessionStart JSON을 stdin으로 직접 전달합니다. native Harness가 Hook을 호출한 검증이 아닙니다. local_hooks/local_traces 등 별도 변수를 사용해 API 기록을 덮어쓰지 않습니다. 전체 실행 원본·오류·프롬프트를 내보내는 기능이 아닙니다.


In [ ]:
if docker_enabled and RUN_LOCAL_CHECKS:
    build_images(root, executor=False)
    with DockerLab("13_hooks_telemetry", root) as lab:
        network, worker = lab.package_environment()
        local_collector, local_endpoint = start_collector(lab, network)
        local_hooks = ApplicationHooks(local_endpoint + "/v1/traces", lab.run_id)
        try:
            local_hooks.emit(
                "report.started", chapter=13, status="running", prompt="DO_NOT_EXPORT_PROMPT"
            )
            lab.install(worker)
            lab.execute(worker, SKILL_COMMAND)
            local_metrics, _ = lab.collect_report(worker)
            local_hooks.emit("report.completed", chapter=13, status="completed")
            payload = {
                "hook_event_name": "SessionStart",
                "session_id": "sess_protocol_replay",
                "prompt": "DO_NOT_EXPORT_PROMPT",
                "cwd": "/private/user/path",
            }
            docker(
                [
                    "exec",
                    "-i",
                    "-e",
                    ("TUTORIAL_OTLP_LOGS_ENDPOINT=http://collector:" "4318/v1/logs"),
                    worker,
                    "python",
                    "/opt/tutorial/native_hooks/session_start.py",
                ],
                input=json.dumps(payload),
            )
        finally:
            local_hooks.close()
        local_traces = read_collector_file(lab, local_collector, "traces.jsonl", needle="report.completed")
        local_logs = read_collector_file(lab, local_collector, "logs.jsonl", needle="SessionStart")
        assert "report.completed" in local_traces and lab.run_id in local_traces
        assert "SessionStart" in local_logs and "sess_protocol_replay" in local_logs
        assert (
            "DO_NOT_EXPORT_PROMPT" not in local_traces + local_logs
            and "/private/user/path" not in local_traces + local_logs
        )
else:
    print("Optional Docker example skipped. Set RUN_DOCKER=True and RUN_LOCAL_CHECKS=True to run it separately.")


## 4. 관리 설정과 허용 필드 확인

이 TOML 예제는 별도 Codex 관리 환경용입니다. Agents API 설정으로 업로드하지 않습니다. native dispatch 지원 여부·정책·Codex 버전은 해당 배포에서 별도로 검증해야 합니다.

In [ ]:
import tomllib

requirements = tomllib.loads((root / "deploy/codex/requirements.toml.example").read_text())
assert requirements["features"]["hooks"] is True
proposed = {"chapter": 13, "status": "completed", "prompt": "do-not-export", "cwd": "/private/path"}
sanitized = {k: v for k, v in proposed.items() if k in ApplicationHooks.ALLOWED}
assert "prompt" not in sanitized and "cwd" not in sanitized
print(sanitized)


## 완료 기준과 다음 장

callback 기록·실제 app span·API 기본 Trace·export·별도 native Hook 프로토콜 재생의 주체와 의미를 구분할 수 있어야 합니다. 기록의 같은 Session/Turn, 실제 수신 여부, None usage와 0, span 시간과 duration_ms, 분석 성공과 export 오류를 나누어 읽습니다.

[14장 관찰자 재연결](14_operations.ipynb)은 원래 Session·Turn·저장된 Items와 파일로 복구 결과를 확인합니다. 이 장의 app OTel과 별도이며 추가 Trace export를 필수로 하지 않습니다. [실습 안내와 검증 범위](../../docs/validation.md)에서 실행·검사 범위를 확인하세요. 실제 Dashboard 화면 확인은 자동 검증에 포함하지 않습니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Observability and usage](https://developers.openai.com/api/docs/guides/agents-api/observability) — 실행 기록·Dashboard·Turn usage와 비용 해석.
- [Tracing](https://developers.openai.com/api/docs/guides/agents-api/tracing) — Session·Turn·Span, Trace 준비 지연과 OTLP JSON export.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.

위 문서는 Agents API 관측에 해당합니다. 애플리케이션 callback·OTel 전송 코드와 별도 Codex Hook 설정은 서로 구분합니다.

### 보조 자료 — Codex 문서

- [Codex Hooks](https://learn.chatgpt.com/docs/hooks) — 별도 Codex 환경의 SessionStart·requirements.toml 설명입니다. 이 노트북의 API 이벤트 callback과 구분하며, stdin 프로토콜 재생을 Agents API native Hook dispatch 검증으로 해석하지 않습니다.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
